# NLP Homework 1: Word2Vec & RNN from Scratch
In this assignment, you will bridge the gap between the theoretical math discussed in the lecture and the practical engineering required to build deep sequential models. You will construct two core architectures from scratch:

1. Simple Skip-Gram Word2Vec
2. Vanilla Recurrent Neural Network (RNN)

## Task 1: Word2Vec (Skip-Gram) Implementation

In the Skip-Gram architecture, we predict context words $w_o$ given a center target word $w_c$.

**Your Task**:
Write the SkipGramSimple class.

1. In `__init__`, define two separate `nn.Embedding` lookup tables: one for the target/center words, and one for the context/outside words.
2. In `forward`, you will receive batches of target token IDs and context token IDs. Extract their respective dense vectors.
3. Compute the raw similarity score using the dot product between the target vector and the context vector.

$$score(c, o) = \sum_{j=1}^{d} v_{c, j} \cdot u_{o, j}$$

(Note: Do not apply a sigmoid activation at the end. We assume the use of BCEWithLogitsLoss later, which handles the sigmoid internally for numerical stability).

### Why `nn.Embedding` instead of One-Hot + `nn.Linear`?
During the lecture, we demonstrated how to extract a word vector by multiplying a one-hot encoded vector by a weight matrix.

If our vocabulary size $V$ is 10,000, and our embedding dimension $d$ is 300, our weight matrix $W$ has a shape of $[10000, 300]$.
A one-hot vector $x$ for a single word has a shape of $[1, 10000]$ (where 9,999 values are exactly $0$).Mathematically, the forward pass is:

$$e = x W$$

**The Engineering Problem**: If you actually implement this using `nn.Linear`, the GPU will perform 3,000,000 floating-point multiplications for a single word. Because 9,999 of the inputs are zero, 2,999,700 of those operations are a complete waste of compute. When processing batches of long sequences, this approach will instantly exhaust your VRAM and grind training to a halt.

**The Solution**: PyTorch provides `nn.Embedding`. Instead of performing dense matrix multiplication, `nn.Embedding` stores the $[10000, 300]$ matrix and performs a direct, $O(1)$ memory index lookup. If you pass it token ID `402`, it immediately grabs row `402` from the matrix.

For this homework, you will use `nn.Embedding` for all vector lookups. Mathematically, this is equivalent to what we defined during the lecture.

In [1]:
import torch
import torch.nn as nn

class SkipGramSimple(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.target_embed = nn.Embedding(vocab_size, embed_dim)
        self.context_embed = nn.Embedding(vocab_size, embed_dim)


    def forward(self, target_ids, context_ids):
        """
        target_ids shape: [batch_size]
        context_ids shape: [batch_size]
        Returns: [batch_size]
        """
        v_c = self.target_embed(target_ids)
        u_o = self.context_embed(context_ids)
        return (v_c * u_o).sum(dim=1)

# --- Sanity Check ---
model_sg = SkipGramSimple(vocab_size=1000, embed_dim=50)
t_dummy = torch.tensor([10, 20, 30])
c_dummy = torch.tensor([11, 25, 32])
assert model_sg(t_dummy, c_dummy).shape == torch.Size([3])

### Feeling the Results: Your Embeddings in Action
You just built the raw mathematical engine of Word2Vec. But does a simple dot product actually learn meaning?

To prove it, we have provided a tiny training loop below. It initializes your SkipGramSimple model and trains it on a micro-corpus of four sentences. Notice how the words "linguistics" and "reconstruction" are used in the exact same contexts (followed by "explores" or "traces").

Run the cell below. If your forward pass is mathematically correct, your model will learn that those two words are semantically related, completely from scratch!

In [2]:
import torch.optim as optim
import torch.nn.functional as F
import random

# 1. A micro-corpus designed to force contextual similarity
text = [
    "linguistics explores language families",
    "reconstruction explores ancient families",
    "linguistics traces language roots",
    "reconstruction traces ancient roots"
]

print("Building vocabulary and generating training pairs...")
words = " ".join(text).split()
vocab = list(set(words))
word_to_id = {w: i for i, w in enumerate(vocab)}

# Generate positive target-context pairs (window size = 1)
pairs = []
for sentence in text:
    tokens = sentence.split()
    for i, token in enumerate(tokens):
        target = word_to_id[token]
        if i > 0: pairs.append((target, word_to_id[tokens[i-1]]))
        if i < len(tokens) - 1: pairs.append((target, word_to_id[tokens[i+1]]))

# 2. Initialize YOUR model
# We use embed_dim=2 so the network is forced to tightly compress the relationships
model = SkipGramSimple(vocab_size=len(vocab), embed_dim=2)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.1)

# 3. Train the model using positive and negative sampling
print("Training SkipGram...")
epochs = 100
for epoch in range(epochs):
    for target, context in pairs:
        optimizer.zero_grad()

        t_tensor = torch.tensor([target])

        # --- Positive Sample ---
        c_tensor = torch.tensor([context])
        pos_score = model(t_tensor, c_tensor)
        loss_pos = criterion(pos_score, torch.tensor([1.0]))

        # --- Negative Sample ---
        # Pick a random word that is NOT the context
        neg_context = random.randint(0, len(vocab)-1)
        neg_tensor = torch.tensor([neg_context])
        neg_score = model(t_tensor, neg_tensor)
        loss_neg = criterion(neg_score, torch.tensor([0.0]))

        # Combine losses and backpropagate
        loss = loss_pos + loss_neg
        loss.backward()
        optimizer.step()

print("Training complete!\n")

# 4. Evaluate the Learned Semantics
embeddings = model.target_embed.weight.data

vec_linguistics = embeddings[word_to_id["linguistics"]]
vec_reconstruction = embeddings[word_to_id["reconstruction"]]
vec_families = embeddings[word_to_id["families"]]

def cos_sim(v1, v2):
    return F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()

print("--- Learned Cosine Similarities ---")
print(f"Similarity (linguistics, reconstruction) -> {cos_sim(vec_linguistics, vec_reconstruction):.4f} [EXPECT HIGH]")
print(f"Similarity (linguistics, families)       -> {cos_sim(vec_linguistics, vec_families):.4f} [EXPECT LOW]")

Building vocabulary and generating training pairs...
Training SkipGram...
Training complete!

--- Learned Cosine Similarities ---
Similarity (linguistics, reconstruction) -> 0.9934 [EXPECT HIGH]
Similarity (linguistics, families)       -> 0.0519 [EXPECT LOW]


## Task 2: Vanilla RNN Cell from Scratch

You will now implement the temporal recurrence relation without using PyTorch's built-in `nn.RNN` or `nn.LSTM` modules.

The forward iteration formula for the hidden state at time step $t$ is:

$$h_t = \tanh(x_t W_{xh}^T + h_{t-1} W_{hh}^T + b_h)$$

**Your Task**:
Write the `VanillaRNNScratch` class.
1. In `__init__`, explicitly define the trainable matrices $W_{xh}$, $W_{hh}$, and the bias vector $b_h$ using `nn.Parameter`. Initialize the weights with small random values (e.g., using `torch.randn() * 0.01`) and the bias with zeros.
2. In `forward`, initialize the starting hidden state $h_0$ as a tensor of zeros.
3. Create a loop to iterate sequentially through the `seq_len` dimension of the input tensor. At each time step, slice the input, compute the new $h_t$ using the formula above, and carry it forward.
4. Implement a Many-to-One architecture: return only the final hidden state $h_T$ after the loop concludes.

In [3]:
class VanillaRNNScratch(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.W_xh = nn.Parameter(torch.randn(hidden_dim, input_dim) * 0.01)
        self.W_hh = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01)
        self.b_h  = nn.Parameter(torch.zeros(hidden_dim))


    def forward(self, x):
        """
        x shape: [batch_size, seq_len, input_dim]
        Returns: [batch_size, hidden_dim]
        """
        batch_size, seq_len, input_dim = x.shape
        h_t = torch.zeros(batch_size, self.hidden_dim, device=x.device)

        for t in range(seq_len):
            x_t = x[:, t, :]                                   # [batch, input_dim]
            h_t = torch.tanh(x_t @ self.W_xh.T + h_t @ self.W_hh.T + self.b_h)

        return h_t   # final hidden state only -> many-to-one
        raise NotImplementedError()

# --- Sanity Check ---
rnn_scratch = VanillaRNNScratch(input_dim=64, hidden_dim=128)
dummy_x = torch.randn(16, 30, 64)
assert rnn_scratch(dummy_x).shape == torch.Size([16, 128])

## Task 3: Integration & End-to-End Training
It is time to prove that your scratch RNN works mathematically. You will wrap your RNN inside a classifier, generate some dummy sequence data, and write a training loop to optimize it.

**Your Task**:

1. Complete the `CustomSentimentClassifier` class. It must contain an `nn.Embedding` layer, your `VanillaRNNScratch` layer, and a final `nn.Linear` classification head.
2. Instantiate your model, the `BCEWithLogitsLoss` criterion, and the `Adam` optimizer.
3. Write a standard PyTorch training loop on the IMDB dataset (provided) for 5 epochs. If your forward pass and parameter definitions in Task 2 are mathematically sound, PyTorch's autograd will successfully backpropagate through your custom loops and the loss will decrease.
4. Use your model to classify the sentiment of some movie reviews written by yourself.

*(Note: you may use the solutions of Part 3 of NLP Practical 1 for this section. They are essentially the same task.)*

### Data Preparation
The code to download the IMDB dataset, build a simple vocabulary, and construct the PyTorch DataLoader is provided below.

In [4]:
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.optim as optim
from collections import Counter
import re

# 1. Load IMDB (Using a small subset for classroom speed)
print("Loading IMDB Dataset...")
dataset = load_dataset("stanfordnlp/imdb")
train_data = dataset['train'].shuffle(seed=42) # you may add `.select(range(N))` to select a subset of the data and speed-up training
test_data = dataset['test'].shuffle(seed=42) # `.select(range(N))`

# 2. Simple Tokenizer & Vocabulary Builder
def tokenize(text):
    return re.sub(r'[^a-z ]+', '', text.lower()).split()

print("Building vocabulary...")
all_words = [word for item in train_data for word in tokenize(item['text'])]
vocab = {word: i+2 for i, (word, _) in enumerate(Counter(all_words).most_common(10000))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

def encode(text):
    return [vocab.get(word, vocab['<UNK>']) for word in tokenize(text)][:250] # Cap at 250 words

# 3. Collate function for DataLoader (Padding)
def collate_fn(batch):
    sequences = [torch.tensor(encode(item['text'])) for item in batch]
    labels = torch.tensor([item['label'] for item in batch]).float()
    padded_seqs = pad_sequence(sequences, batch_first=True, padding_value=0)
    return padded_seqs, labels

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=collate_fn)
print("DataLoader ready!")

Loading IMDB Dataset...


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Building vocabulary...
DataLoader ready!


### Architecture and Model Training

In [5]:
import torch.optim as optim

class CustomSentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = VanillaRNNScratch(embed_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        """
        x shape: [batch_size, seq_len]
        Returns: [batch_size]
        """
        embedded = self.embedding(x)        # [batch, seq_len, embed_dim]
        h_final = self.rnn(embedded)        # [batch, hidden_dim]
        logits = self.fc(h_final).squeeze(1)  # [batch]
        return logits
        raise NotImplementedError()

# -------------------------------------------

# TODO: 1. Instantiate model, criterion, and optimizer
vocab_size = len(vocab)
embed_dim = 64
hidden_dim = 128

model = CustomSentimentClassifier(vocab_size, embed_dim, hidden_dim)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# TODO: 2. Write the training loop for 5 epochs and print the loss

epochs = 5
for epoch in range(epochs):
    total_loss = 0.0
    for sequences, labels in train_loader:
        optimizer.zero_grad()
        logits = model(sequences)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}")



# TODO: 3. Make some predictions on reviews written by yourself.

def predict_sentiment(text, model, vocab):
    model.eval()
    with torch.no_grad():
        ids = torch.tensor([encode(text)])
        logit = model(ids)
        prob = torch.sigmoid(logit).item()
    return ("Positive" if prob > 0.5 else "Negative"), prob

reviews = [
    "This movie was absolutely fantastic, I loved every minute of it.",
    "Terrible film, a complete waste of time."
]
for r in reviews:
    label, prob = predict_sentiment(r, model, vocab)
    print(f"{r!r} -> {label} ({prob:.3f})")

Epoch 1/5 — Loss: 0.6962
Epoch 2/5 — Loss: 0.6959
Epoch 3/5 — Loss: 0.6960
Epoch 4/5 — Loss: 0.6952
Epoch 5/5 — Loss: 0.6952
'This movie was absolutely fantastic, I loved every minute of it.' -> Negative (0.491)
'Terrible film, a complete waste of time.' -> Positive (0.500)
